# Chapter 4.4: Custom CUDA Kernels in vLLM

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the role of custom CUDA kernels in vLLM's performance
2. **Analyze** PagedAttention v1 and v2 kernels in detail
3. **Explain** thread block organization and memory access patterns
4. **Trace** through cache copy/swap and activation kernels
5. **Understand** rotary embedding implementation
6. **Know** how kernels are compiled via torch extensions
7. **Profile** kernel performance characteristics

---

## Prerequisites
- Basic CUDA programming concepts (threads, blocks, warps)
- Understanding of transformer attention mechanism
- Chapter 4.1 (KV-Cache layout, PagedAttention concept)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part4/chapter_4.4_cuda_kernels.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part4/chapter_4.4_cuda_kernels.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Overview of Custom Kernels in vLLM

vLLM implements several custom CUDA kernels for operations that are either:
1. **Not available** in standard libraries (PagedAttention)
2. **Optimized** beyond what PyTorch/cuBLAS provides (fused kernels)
3. **Specific** to vLLM's memory management (cache operations)

### Kernel Categories

```
vllm/csrc/
├── attention/
│   ├── paged_attention_v1.cu    # PagedAttention v1 (single-pass)
│   ├── paged_attention_v2.cu    # PagedAttention v2 (two-pass, larger blocks)
│   └── attention_kernels.cu     # Shared attention utilities
├── cache_kernels.cu             # Block copy, swap, reshape_and_cache
├── activation_kernels.cu        # SiLU, GeGLU, fused activations
├── layernorm_kernels.cu         # RMSNorm, LayerNorm
├── pos_encoding_kernels.cu      # Rotary Position Embedding
├── quantization/                # FP8, INT8, GPTQ, AWQ kernels
└── reduction_kernels.cu         # Cross-block reductions
```

### Why Custom Kernels?

| Operation | Standard Library | vLLM Custom | Reason |
|-----------|-----------------|-------------|--------|
| Attention | FlashAttention | PagedAttention | Paged memory layout |
| Cache write | torch.index_copy | reshape_and_cache | Fused, block-aware |
| RMSNorm | PyTorch | Fused kernel | Avoids extra memory reads |
| RoPE | Manual compute | Fused kernel | Combines with QKV projection |
| Activation | F.silu | Fused SiLU+mul | Saves memory bandwidth |

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from typing import List, Tuple, Optional
import time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. PagedAttention v1: Detailed Walkthrough

### The Problem

Standard attention implementations (like FlashAttention) expect KV-cache in **contiguous** memory. PagedAttention must handle **non-contiguous** blocks scattered across GPU memory.

### Algorithm Overview

```
PagedAttention v1 (single-pass):

For each query token q_i:
  1. Look up block table to find physical blocks
  2. For each KV-cache block:
     a. Load K block from scattered memory location
     b. Compute attention scores: score = q_i @ K_block^T / sqrt(d)
     c. Track running max for numerical stability
  3. Compute softmax using online softmax trick
  4. For each KV-cache block:
     a. Load V block
     b. Accumulate weighted sum: output += softmax_score * V_block
  5. Write output
```

### Thread Block Organization

```
GPU Grid:
  grid_x = num_seqs × num_heads    (one block per sequence-head pair)
  
Thread Block (for PagedAttention v1):
  ┌─────────────────────────────────────────────┐
  │ Warp 0: Processes head_dim elements [0:32]  │
  │ Warp 1: Processes head_dim elements [32:64] │
  │ Warp 2: Processes head_dim elements [64:96] │
  │ Warp 3: Processes head_dim elements [96:128]│
  └─────────────────────────────────────────────┘
  Total threads per block = NUM_WARPS × WARP_SIZE = 4 × 32 = 128

Each thread block computes attention for ONE query, ONE head.
It iterates over ALL KV-cache blocks for that sequence.
```

In [ ]:
def paged_attention_v1_reference(
    query: torch.Tensor,          # [num_seqs, num_heads, head_dim]
    key_cache: torch.Tensor,      # [num_blocks, block_size, num_kv_heads, head_dim]
    value_cache: torch.Tensor,    # [num_blocks, block_size, num_kv_heads, head_dim]
    block_tables: torch.Tensor,   # [num_seqs, max_num_blocks]
    context_lens: torch.Tensor,   # [num_seqs]
    scale: float,
    block_size: int,
) -> torch.Tensor:
    """
    Reference implementation of PagedAttention v1.
    
    This Python code mirrors the logic in:
    vllm/csrc/attention/paged_attention_v1.cu
    
    Note: This is the algorithmic logic, not the CUDA parallelization.
    The actual CUDA kernel parallelizes across sequences and heads.
    """
    num_seqs, num_heads, head_dim = query.shape
    num_kv_heads = key_cache.shape[2]
    num_queries_per_kv = num_heads // num_kv_heads  # GQA ratio
    
    output = torch.zeros_like(query)
    
    for seq_idx in range(num_seqs):
        context_len = context_lens[seq_idx].item()
        num_blocks = (context_len + block_size - 1) // block_size
        
        for head_idx in range(num_heads):
            kv_head_idx = head_idx // num_queries_per_kv
            q = query[seq_idx, head_idx]  # [head_dim]
            
            # Phase 1: Compute attention scores for all KV tokens
            all_scores = []
            all_values = []
            
            for block_idx in range(num_blocks):
                physical_block = block_tables[seq_idx, block_idx].item()
                
                # Determine how many tokens in this block
                start_token = block_idx * block_size
                end_token = min(start_token + block_size, context_len)
                num_tokens = end_token - start_token
                
                # Load K from scattered block
                k_block = key_cache[physical_block, :num_tokens, kv_head_idx]  # [num_tokens, head_dim]
                v_block = value_cache[physical_block, :num_tokens, kv_head_idx]  # [num_tokens, head_dim]
                
                # Compute scores: q @ K^T
                scores = torch.matmul(k_block, q) * scale  # [num_tokens]
                all_scores.append(scores)
                all_values.append(v_block)
            
            # Phase 2: Softmax and weighted sum
            all_scores = torch.cat(all_scores)  # [context_len]
            all_values = torch.cat(all_values)  # [context_len, head_dim]
            
            # Softmax
            attn_weights = F.softmax(all_scores, dim=0)  # [context_len]
            
            # Weighted sum of values
            output[seq_idx, head_idx] = torch.matmul(attn_weights, all_values)  # [head_dim]
    
    return output

# Test the reference implementation
torch.manual_seed(42)

num_seqs = 2
num_heads = 8
num_kv_heads = 2  # GQA: 4 query heads per KV head
head_dim = 64
block_size = 4
num_blocks_total = 20

query = torch.randn(num_seqs, num_heads, head_dim)
key_cache = torch.randn(num_blocks_total, block_size, num_kv_heads, head_dim)
value_cache = torch.randn(num_blocks_total, block_size, num_kv_heads, head_dim)
block_tables = torch.tensor([[0, 3, 7], [1, 5, 0]])  # Block mappings
context_lens = torch.tensor([10, 8])
scale = 1.0 / math.sqrt(head_dim)

output = paged_attention_v1_reference(
    query, key_cache, value_cache, block_tables, context_lens, scale, block_size
)
print(f"Output shape: {output.shape}")
print(f"Output sample: {output[0, 0, :8]}")

## 3. PagedAttention v1: CUDA Kernel Analysis

### Source Code: `vllm/csrc/attention/paged_attention_v1.cu`

```cpp
// Simplified version of the actual CUDA kernel
template<typename scalar_t, int HEAD_SIZE, int BLOCK_SIZE, int NUM_THREADS>
__global__ void paged_attention_v1_kernel(
    scalar_t* __restrict__ out,           // [num_seqs, num_heads, head_dim]
    const scalar_t* __restrict__ q,       // [num_seqs, num_heads, head_dim]
    const scalar_t* __restrict__ k_cache, // [num_blocks, block_size, num_kv_heads, head_dim]
    const scalar_t* __restrict__ v_cache, // [num_blocks, block_size, num_kv_heads, head_dim]
    const int* __restrict__ block_tables, // [num_seqs, max_blocks_per_seq]
    const int* __restrict__ context_lens, // [num_seqs]
    const float scale,
    const int max_blocks_per_seq,
    const int num_kv_heads,
    const int kv_head_stride) {
    
    // Each thread block handles one (sequence, head) pair
    const int seq_idx = blockIdx.x;
    const int head_idx = blockIdx.y;
    const int thread_idx = threadIdx.x;
    const int warp_idx = thread_idx / WARP_SIZE;
    const int lane = thread_idx % WARP_SIZE;
    
    // Map query head to KV head (for GQA)
    const int kv_head_idx = head_idx / (num_heads / num_kv_heads);
    
    const int context_len = context_lens[seq_idx];
    const int num_blocks = DIVIDE_ROUND_UP(context_len, BLOCK_SIZE);
    
    // Load query vector into shared memory
    __shared__ float q_shared[HEAD_SIZE];
    // Each thread loads part of the query
    for (int i = thread_idx; i < HEAD_SIZE; i += NUM_THREADS) {
        q_shared[i] = (float)q[seq_idx * num_heads * HEAD_SIZE + 
                               head_idx * HEAD_SIZE + i];
    }
    __syncthreads();
    
    // Online softmax variables
    float max_score = -INFINITY;
    float sum_exp = 0.0f;
    float output_local[HEAD_SIZE / NUM_THREADS];  // Per-thread output accumulator
    // Initialize output to zero
    for (int i = 0; i < HEAD_SIZE / NUM_THREADS; i++) {
        output_local[i] = 0.0f;
    }
    
    // Iterate over all KV-cache blocks for this sequence
    for (int block_idx = 0; block_idx < num_blocks; block_idx++) {
        // Look up physical block ID from block table
        int physical_block = block_tables[seq_idx * max_blocks_per_seq + block_idx];
        
        // Compute attention scores for this block
        for (int token_idx = 0; token_idx < BLOCK_SIZE; token_idx++) {
            int global_token_idx = block_idx * BLOCK_SIZE + token_idx;
            if (global_token_idx >= context_len) break;
            
            // Compute dot product: q @ k
            float score = 0.0f;
            for (int d = thread_idx; d < HEAD_SIZE; d += NUM_THREADS) {
                float k_val = (float)k_cache[
                    physical_block * BLOCK_SIZE * num_kv_heads * HEAD_SIZE +
                    token_idx * num_kv_heads * HEAD_SIZE +
                    kv_head_idx * HEAD_SIZE + d];
                score += q_shared[d] * k_val;
            }
            score *= scale;
            
            // Warp-level reduction for the score
            score = warp_reduce_sum(score);  // Sum across threads in warp
            // Cross-warp reduction via shared memory
            // ...
            
            // Online softmax update
            float old_max = max_score;
            max_score = fmaxf(max_score, score);
            float exp_score = expf(score - max_score);
            float correction = expf(old_max - max_score);
            sum_exp = sum_exp * correction + exp_score;
            
            // Update output accumulator with corrected weights
            for (int d = thread_idx; d < HEAD_SIZE; d += NUM_THREADS) {
                float v_val = (float)v_cache[
                    physical_block * BLOCK_SIZE * num_kv_heads * HEAD_SIZE +
                    token_idx * num_kv_heads * HEAD_SIZE +
                    kv_head_idx * HEAD_SIZE + d];
                output_local[d / NUM_THREADS] = 
                    output_local[d / NUM_THREADS] * correction + 
                    exp_score * v_val;
            }
        }
    }
    
    // Normalize by sum of exponentials
    for (int d = thread_idx; d < HEAD_SIZE; d += NUM_THREADS) {
        out[seq_idx * num_heads * HEAD_SIZE + head_idx * HEAD_SIZE + d] = 
            (scalar_t)(output_local[d / NUM_THREADS] / sum_exp);
    }
}
```

In [ ]:
def paged_attention_v1_cuda_sim(
    query: torch.Tensor,
    key_cache: torch.Tensor,
    value_cache: torch.Tensor,
    block_tables: torch.Tensor,
    context_lens: torch.Tensor,
    scale: float,
    block_size: int,
    num_threads: int = 128,
) -> torch.Tensor:
    """
    Simulates the CUDA kernel's per-thread-block execution.
    
    Uses ONLINE SOFTMAX (Milakov & Gimelshein, 2018) to avoid
    materializing the full attention matrix.
    """
    num_seqs, num_heads, head_dim = query.shape
    num_kv_heads = key_cache.shape[2]
    num_queries_per_kv = num_heads // num_kv_heads
    output = torch.zeros_like(query)
    
    for seq_idx in range(num_seqs):
        for head_idx in range(num_heads):
            kv_head_idx = head_idx // num_queries_per_kv
            q = query[seq_idx, head_idx]  # [head_dim]
            context_len = context_lens[seq_idx].item()
            num_blocks = (context_len + block_size - 1) // block_size
            
            # === Online Softmax State ===
            max_score = float('-inf')
            sum_exp = 0.0
            acc = torch.zeros(head_dim)  # Output accumulator
            
            # === Iterate over KV blocks (like CUDA thread block loop) ===
            for blk_idx in range(num_blocks):
                phys_block = block_tables[seq_idx, blk_idx].item()
                start = blk_idx * block_size
                end = min(start + block_size, context_len)
                
                for tok_idx in range(end - start):
                    # Load K token from scattered memory
                    k = key_cache[phys_block, tok_idx, kv_head_idx]  # [head_dim]
                    
                    # Dot product (distributed across threads in CUDA)
                    score = (q * k).sum().item() * scale
                    
                    # === Online softmax update ===
                    old_max = max_score
                    max_score = max(max_score, score)
                    
                    correction = math.exp(old_max - max_score) if old_max != float('-inf') else 0.0
                    exp_score = math.exp(score - max_score)
                    
                    # Correct previous accumulation and add new
                    sum_exp = sum_exp * correction + exp_score
                    acc = acc * correction + exp_score * value_cache[phys_block, tok_idx, kv_head_idx]
            
            # Normalize
            output[seq_idx, head_idx] = acc / sum_exp
    
    return output

# Verify online softmax matches standard softmax
output_ref = paged_attention_v1_reference(
    query, key_cache, value_cache, block_tables, context_lens, scale, block_size)
output_online = paged_attention_v1_cuda_sim(
    query, key_cache, value_cache, block_tables, context_lens, scale, block_size)

diff = (output_ref - output_online).abs().max().item()
print(f"Max difference between reference and online softmax: {diff:.8f}")
print(f"Outputs match: {diff < 1e-5}")

## 4. PagedAttention v2: Two-Pass Algorithm

### Why v2?

PagedAttention v1 has a problem: a single thread block processes ALL KV blocks for a sequence. For long sequences, this means:
- Each thread block does a lot of work
- Poor GPU utilization (not enough thread blocks to saturate all SMs)

### v2: Partition Across Thread Blocks

```
v1: One thread block per (seq, head)
┌──────────────────────────────────────────────┐
│ Thread Block 0: processes ALL blocks for seq0 │
│   Block 0 → Block 1 → ... → Block N          │
└──────────────────────────────────────────────┘

v2: Multiple thread blocks per (seq, head)
┌──────────────────────┐ ┌──────────────────────┐
│ TB 0: Blocks 0-3     │ │ TB 1: Blocks 4-7     │
│ Partial result + max │ │ Partial result + max │
└──────────────────────┘ └──────────────────────┘
              │                      │
              ▼                      ▼
         ┌─────────────────────────────────┐
         │  REDUCE: Combine partial results │
         │  using log-sum-exp correction    │
         └─────────────────────────────────┘
```

### Source: `vllm/csrc/attention/paged_attention_v2.cu`

```cpp
// Pass 1: Compute partial results
__global__ void paged_attention_v2_kernel(
    float* __restrict__ exp_sums,       // [num_seqs, num_heads, max_num_partitions]
    float* __restrict__ max_logits,      // [num_seqs, num_heads, max_num_partitions]
    scalar_t* __restrict__ tmp_out,      // [num_seqs, num_heads, max_num_partitions, head_dim]
    // ... same as v1
) {
    const int partition_idx = blockIdx.z;  // NEW: partition dimension
    // Each partition handles a subset of KV blocks
    // ...
}

// Pass 2: Reduce partial results
__global__ void paged_attention_v2_reduce_kernel(
    scalar_t* __restrict__ out,
    const float* __restrict__ exp_sums,
    const float* __restrict__ max_logits,
    const scalar_t* __restrict__ tmp_out,
    // ...
) {
    // Combine partial softmax results using log-sum-exp
    // ...
}
```

In [ ]:
def paged_attention_v2_reference(
    query: torch.Tensor,
    key_cache: torch.Tensor,
    value_cache: torch.Tensor,
    block_tables: torch.Tensor,
    context_lens: torch.Tensor,
    scale: float,
    block_size: int,
    partition_size: int = 4,  # KV blocks per partition
) -> torch.Tensor:
    """
    Reference implementation of PagedAttention v2 (two-pass).
    
    Pass 1: Compute partial attention for each partition
    Pass 2: Reduce partial results using log-sum-exp
    """
    num_seqs, num_heads, head_dim = query.shape
    num_kv_heads = key_cache.shape[2]
    num_queries_per_kv = num_heads // num_kv_heads
    output = torch.zeros_like(query)
    
    for seq_idx in range(num_seqs):
        for head_idx in range(num_heads):
            kv_head_idx = head_idx // num_queries_per_kv
            q = query[seq_idx, head_idx]
            context_len = context_lens[seq_idx].item()
            num_blocks = (context_len + block_size - 1) // block_size
            num_partitions = (num_blocks + partition_size - 1) // partition_size
            
            # === PASS 1: Partial results per partition ===
            partial_outputs = []
            partial_max_scores = []
            partial_sum_exps = []
            
            for part_idx in range(num_partitions):
                start_blk = part_idx * partition_size
                end_blk = min(start_blk + partition_size, num_blocks)
                
                # Online softmax within this partition
                max_score = float('-inf')
                sum_exp = 0.0
                acc = torch.zeros(head_dim)
                
                for blk_idx in range(start_blk, end_blk):
                    phys_block = block_tables[seq_idx, blk_idx].item()
                    start = blk_idx * block_size
                    end = min(start + block_size, context_len)
                    
                    for tok_idx in range(end - start):
                        k = key_cache[phys_block, tok_idx, kv_head_idx]
                        score = (q * k).sum().item() * scale
                        
                        old_max = max_score
                        max_score = max(max_score, score)
                        correction = math.exp(old_max - max_score) if old_max != float('-inf') else 0.0
                        exp_score = math.exp(score - max_score)
                        
                        sum_exp = sum_exp * correction + exp_score
                        acc = acc * correction + exp_score * value_cache[phys_block, tok_idx, kv_head_idx]
                
                partial_outputs.append(acc)
                partial_max_scores.append(max_score)
                partial_sum_exps.append(sum_exp)
            
            # === PASS 2: Reduce partitions ===
            # Find global max
            global_max = max(partial_max_scores)
            
            # Correct and combine
            final_output = torch.zeros(head_dim)
            final_sum_exp = 0.0
            
            for i in range(num_partitions):
                correction = math.exp(partial_max_scores[i] - global_max)
                corrected_sum = partial_sum_exps[i] * correction
                final_sum_exp += corrected_sum
                final_output += partial_outputs[i] * correction
            
            output[seq_idx, head_idx] = final_output / final_sum_exp
    
    return output

# Verify v2 matches v1
output_v2 = paged_attention_v2_reference(
    query, key_cache, value_cache, block_tables, context_lens, scale, block_size)

diff = (output_ref - output_v2).abs().max().item()
print(f"Max difference v1 vs v2: {diff:.8f}")
print(f"v1 and v2 produce identical results: {diff < 1e-5}")

## 5. Memory Access Patterns

### Why Memory Access Patterns Matter

GPU HBM bandwidth is the primary bottleneck for decode attention. The key challenge is that paged blocks are **scattered** in memory:

```
Contiguous access (FlashAttention):
  Memory: [K0][K1][K2][K3][K4][K5][K6][K7]...
  Read:   ████████████████████████████████
  → Coalesced, maximum bandwidth utilization

Scattered access (PagedAttention):
  Memory: [K0][__][K4][__][__][K1][__][K5]...
  Read:   ████      ████         ████  ████
  → Non-coalesced, reduced bandwidth
  
Mitigation: Block size = 16 ensures WITHIN a block,
  access is coalesced. Only BETWEEN blocks is scattered.
```

In [ ]:
def analyze_memory_access_pattern(
    block_tables: torch.Tensor,
    context_lens: torch.Tensor,
    block_size: int,
    num_kv_heads: int,
    head_dim: int,
    dtype_bytes: int = 2,
):
    """
    Analyze the memory access pattern of PagedAttention.
    """
    num_seqs = block_tables.shape[0]
    bytes_per_block = block_size * num_kv_heads * head_dim * dtype_bytes
    
    print("=== Memory Access Pattern Analysis ===")
    print(f"Block size: {block_size}")
    print(f"Bytes per KV block (K or V): {bytes_per_block:,}")
    
    for seq_idx in range(num_seqs):
        ctx_len = context_lens[seq_idx].item()
        num_blocks = (ctx_len + block_size - 1) // block_size
        blocks = block_tables[seq_idx, :num_blocks].tolist()
        
        print(f"\nSequence {seq_idx} ({ctx_len} tokens, {num_blocks} blocks):")
        print(f"  Block table: {blocks}")
        
        # Check if blocks are contiguous
        is_contiguous = all(blocks[i] + 1 == blocks[i+1] for i in range(len(blocks)-1))
        print(f"  Contiguous: {is_contiguous}")
        
        # Calculate address jumps
        jumps = [abs(blocks[i+1] - blocks[i]) for i in range(len(blocks)-1)]
        if jumps:
            print(f"  Block jumps: {jumps}")
            print(f"  Max jump: {max(jumps)} blocks = {max(jumps) * bytes_per_block:,} bytes")
            print(f"  Avg jump: {np.mean(jumps):.1f} blocks")
        
        # Total data read
        total_read = num_blocks * bytes_per_block * 2  # K + V
        print(f"  Total KV data read: {total_read:,} bytes = {total_read/1024:.1f} KB")

analyze_memory_access_pattern(
    block_tables, context_lens, block_size,
    num_kv_heads=2, head_dim=64, dtype_bytes=2
)

## 6. Cache Operations Kernels

### `reshape_and_cache` Kernel

This kernel writes new KV entries to the cache during attention computation:

```cpp
// From vllm/csrc/cache_kernels.cu
__global__ void reshape_and_cache_kernel(
    const scalar_t* __restrict__ key,       // [num_tokens, num_heads, head_dim]
    const scalar_t* __restrict__ value,      // [num_tokens, num_heads, head_dim]
    scalar_t* __restrict__ key_cache,        // [num_blocks, block_size, num_heads, head_dim]
    scalar_t* __restrict__ value_cache,      // [num_blocks, block_size, num_heads, head_dim]
    const int64_t* __restrict__ slot_mapping, // [num_tokens]
    const int block_size,
    const int num_heads,
    const int head_size) {
    
    const int token_idx = blockIdx.x;      // One block per token
    const int64_t slot = slot_mapping[token_idx];
    
    // Compute physical location
    const int block_idx = slot / block_size;
    const int block_offset = slot % block_size;
    
    // Copy key and value to cache
    // Each thread copies one element
    const int n = num_heads * head_size;
    for (int i = threadIdx.x; i < n; i += blockDim.x) {
        const int head_idx = i / head_size;
        const int head_offset = i % head_size;
        
        // Source: token's KV projection output
        const int src_idx = token_idx * n + i;
        
        // Destination: cache slot
        const int dst_idx = block_idx * block_size * n + 
                            block_offset * n + i;
        
        key_cache[dst_idx] = key[src_idx];
        value_cache[dst_idx] = value[src_idx];
    }
}
```

In [ ]:
def reshape_and_cache_reference(
    key: torch.Tensor,           # [num_tokens, num_heads, head_dim]
    value: torch.Tensor,         # [num_tokens, num_heads, head_dim]
    key_cache: torch.Tensor,     # [num_blocks, block_size, num_heads, head_dim]
    value_cache: torch.Tensor,   # [num_blocks, block_size, num_heads, head_dim]
    slot_mapping: torch.Tensor,  # [num_tokens]
    block_size: int,
):
    """
    Reference implementation of reshape_and_cache kernel.
    
    This writes new KV entries into the paged cache.
    Each token's KV projection output goes to a specific slot
    determined by the block table.
    """
    for token_idx in range(key.shape[0]):
        slot = slot_mapping[token_idx].item()
        block_idx = slot // block_size
        block_offset = slot % block_size
        
        key_cache[block_idx, block_offset] = key[token_idx]
        value_cache[block_idx, block_offset] = value[token_idx]

# Demo
num_tokens = 5
num_heads = 4
head_dim = 32
block_size = 4
num_blocks = 10

key = torch.randn(num_tokens, num_heads, head_dim)
value = torch.randn(num_tokens, num_heads, head_dim)
key_cache = torch.zeros(num_blocks, block_size, num_heads, head_dim)
value_cache = torch.zeros(num_blocks, block_size, num_heads, head_dim)

# Slot mapping: token i goes to slot slot_mapping[i]
# slot = block_id * block_size + offset
slot_mapping = torch.tensor([5*4+0, 5*4+1, 5*4+2, 5*4+3, 8*4+0])  # Block 5 (4 tokens) + Block 8 (1 token)

print("Before reshape_and_cache:")
print(f"  Key cache block 5 sum: {key_cache[5].abs().sum():.4f}")

reshape_and_cache_reference(key, value, key_cache, value_cache, slot_mapping, block_size)

print("\nAfter reshape_and_cache:")
print(f"  Key cache block 5 sum: {key_cache[5].abs().sum():.4f}")
print(f"  Key cache block 8 sum: {key_cache[8].abs().sum():.4f}")
print(f"  Verification: key[0] matches cache[5][0]: {torch.allclose(key[0], key_cache[5, 0])}")

## 7. Activation Kernels: SiLU and Fused Operations

### SiLU (Swish) Activation

LLaMA and many modern models use SiLU activation in the MLP:
```
MLP(x) = down_proj(SiLU(gate_proj(x)) * up_proj(x))
```

vLLM fuses the SiLU activation with the element-wise multiplication:

```cpp
// From vllm/csrc/activation_kernels.cu
__global__ void silu_and_mul_kernel(
    scalar_t* __restrict__ out,          // [num_tokens, hidden_size]
    const scalar_t* __restrict__ input,  // [num_tokens, 2 * hidden_size]
    const int hidden_size) {
    
    const int token_idx = blockIdx.x;
    
    for (int i = threadIdx.x; i < hidden_size; i += blockDim.x) {
        // First half: gate (SiLU)
        float x = (float)input[token_idx * 2 * hidden_size + i];
        // Second half: up
        float y = (float)input[token_idx * 2 * hidden_size + hidden_size + i];
        
        // SiLU(x) * y
        out[token_idx * hidden_size + i] = 
            (scalar_t)(x / (1.0f + expf(-x)) * y);
    }
}
```

### Why Fuse?

Without fusion:
1. Read gate_output (HBM → registers): bandwidth cost
2. Compute SiLU, write back: bandwidth cost
3. Read SiLU_output and up_output: bandwidth cost
4. Multiply, write: bandwidth cost

With fusion:
1. Read gate_output and up_output once: bandwidth cost
2. Compute SiLU * mul, write: bandwidth cost

**Saves ~50% memory bandwidth for this operation.**

In [ ]:
def silu_and_mul_naive(gate_output: torch.Tensor, up_output: torch.Tensor) -> torch.Tensor:
    """Naive (unfused) SiLU activation with multiply."""
    silu = F.silu(gate_output)  # SiLU(x) = x * sigmoid(x)
    return silu * up_output

def silu_and_mul_fused(combined: torch.Tensor) -> torch.Tensor:
    """
    Fused SiLU + multiply.
    Input is [gate_output, up_output] concatenated along last dim.
    Mirrors vLLM's fused kernel.
    """
    hidden_size = combined.shape[-1] // 2
    gate = combined[..., :hidden_size]
    up = combined[..., hidden_size:]
    return F.silu(gate) * up

# Benchmark
num_tokens = 256
hidden_size = 4096

gate = torch.randn(num_tokens, hidden_size)
up = torch.randn(num_tokens, hidden_size)
combined = torch.cat([gate, up], dim=-1)

# Verify correctness
out_naive = silu_and_mul_naive(gate, up)
out_fused = silu_and_mul_fused(combined)
print(f"SiLU+mul outputs match: {torch.allclose(out_naive, out_fused, atol=1e-6)}")

# Memory bandwidth analysis
dtype_bytes = 4  # float32
naive_bytes = (hidden_size * 3 + hidden_size) * dtype_bytes * num_tokens  # 3 reads + 1 write per op
fused_bytes = (hidden_size * 2 + hidden_size) * dtype_bytes * num_tokens  # 2 reads + 1 write
print(f"\nMemory bandwidth:")
print(f"  Naive: {naive_bytes / 1024 / 1024:.1f} MB")
print(f"  Fused: {fused_bytes / 1024 / 1024:.1f} MB")
print(f"  Saving: {(1 - fused_bytes/naive_bytes)*100:.0f}%")

## 8. RMSNorm Kernel

### RMSNorm Formula

```
RMSNorm(x) = x * weight / sqrt(mean(x^2) + eps)
```

### Fused Kernel

```cpp
// From vllm/csrc/layernorm_kernels.cu (simplified)
__global__ void rms_norm_kernel(
    scalar_t* __restrict__ out,
    const scalar_t* __restrict__ input,
    const scalar_t* __restrict__ weight,
    const float eps,
    const int hidden_size) {
    
    // One thread block per token (row)
    const int token_idx = blockIdx.x;
    
    // Step 1: Compute sum of squares (reduction)
    float sum_sq = 0.0f;
    for (int i = threadIdx.x; i < hidden_size; i += blockDim.x) {
        float x = (float)input[token_idx * hidden_size + i];
        sum_sq += x * x;
    }
    
    // Warp-level reduction
    sum_sq = block_reduce_sum(sum_sq);
    
    // Step 2: Compute normalization factor
    __shared__ float rms_inv;
    if (threadIdx.x == 0) {
        rms_inv = rsqrtf(sum_sq / hidden_size + eps);
    }
    __syncthreads();
    
    // Step 3: Normalize and scale
    for (int i = threadIdx.x; i < hidden_size; i += blockDim.x) {
        float x = (float)input[token_idx * hidden_size + i];
        out[token_idx * hidden_size + i] = 
            (scalar_t)(x * rms_inv * (float)weight[i]);
    }
}
```

In [ ]:
def rms_norm_reference(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6):
    """
    Reference RMSNorm implementation.
    """
    # Compute RMS
    rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    # Normalize and scale
    return x / rms * weight

def rms_norm_fused_sim(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6):
    """
    Simulates the fused CUDA kernel (single pass over data).
    """
    # In the CUDA kernel, this is done in a single pass:
    # 1. Compute sum of squares (reduction)
    # 2. Compute rsqrt
    # 3. Multiply by weight
    # All without writing intermediate results to global memory
    
    variance = x.pow(2).mean(-1, keepdim=True)
    rms_inv = torch.rsqrt(variance + eps)
    return x * rms_inv * weight

# Test
hidden_size = 4096
x = torch.randn(32, hidden_size)
weight = torch.ones(hidden_size)

out_ref = rms_norm_reference(x, weight)
out_fused = rms_norm_fused_sim(x, weight)

print(f"RMSNorm outputs match: {torch.allclose(out_ref, out_fused, atol=1e-6)}")

# Memory analysis
# Naive: read x (1), write x^2 (1), read x^2 for mean (1), write mean (1), 
#        read x and mean (2), write norm (1), read norm and weight (2), write out (1) = 9 passes
# Fused: read x (1), internal reduction, read weight (1), write out (1) = 3 passes
print(f"\nMemory passes:")
print(f"  PyTorch naive: ~5 global memory passes")
print(f"  Fused CUDA: 2 global memory passes (read x+weight, write output)")
print(f"  Bandwidth saving: ~60%")

## 9. Rotary Position Embedding (RoPE) Kernel

### RoPE Formula

```
RoPE applies rotation to pairs of dimensions:

For dimensions (2i, 2i+1):
  theta_i = 10000^(-2i/d)
  
  [q'_{2i}  ]   [cos(m*theta_i)  -sin(m*theta_i)] [q_{2i}  ]
  [q'_{2i+1}] = [sin(m*theta_i)   cos(m*theta_i)] [q_{2i+1}]

where m is the position index.
```

In [ ]:
def rotary_embedding_reference(
    positions: torch.Tensor,  # [num_tokens]
    query: torch.Tensor,      # [num_tokens, num_heads, head_dim]
    key: torch.Tensor,        # [num_tokens, num_kv_heads, head_dim]
    head_dim: int,
    base: float = 10000.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Reference implementation of Rotary Position Embedding.
    
    Source: vllm/csrc/pos_encoding_kernels.cu
    """
    # Compute frequency bases
    # theta_i = base^(-2i/d) for i in [0, d/2)
    dim_pairs = head_dim // 2
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    
    # Compute rotation angles: position * frequency
    # [num_tokens, dim_pairs]
    angles = positions.unsqueeze(-1).float() * freqs.unsqueeze(0)
    
    cos = torch.cos(angles)  # [num_tokens, dim_pairs]
    sin = torch.sin(angles)  # [num_tokens, dim_pairs]
    
    def apply_rope(x):
        """Apply rotation to query or key."""
        # Split into even and odd dimensions
        x_even = x[..., 0::2]  # [num_tokens, num_heads, dim_pairs]
        x_odd = x[..., 1::2]   # [num_tokens, num_heads, dim_pairs]
        
        # Expand cos/sin for broadcasting
        c = cos.unsqueeze(1)  # [num_tokens, 1, dim_pairs]
        s = sin.unsqueeze(1)  # [num_tokens, 1, dim_pairs]
        
        # Apply rotation
        out_even = x_even * c - x_odd * s
        out_odd = x_even * s + x_odd * c
        
        # Interleave back
        out = torch.stack([out_even, out_odd], dim=-1).flatten(-2)
        return out
    
    return apply_rope(query), apply_rope(key)

# Demo
num_tokens = 4
num_heads = 8
num_kv_heads = 2
head_dim = 64

positions = torch.tensor([0, 1, 2, 3])
q = torch.randn(num_tokens, num_heads, head_dim)
k = torch.randn(num_tokens, num_kv_heads, head_dim)

q_rot, k_rot = rotary_embedding_reference(positions, q, k, head_dim)

print(f"Query shape: {q.shape} -> {q_rot.shape}")
print(f"Key shape: {k.shape} -> {k_rot.shape}")

# Verify: RoPE preserves norms (approximately)
q_norm_before = q.norm(dim=-1).mean().item()
q_norm_after = q_rot.norm(dim=-1).mean().item()
print(f"\nQuery norm preservation: {q_norm_before:.4f} -> {q_norm_after:.4f}")
print(f"RoPE is a rotation (norm-preserving): {abs(q_norm_before - q_norm_after) < 0.01}")

## 10. Kernel Compilation: torch Extensions

### How vLLM Compiles Custom Kernels

vLLM uses PyTorch's C++ extension mechanism to compile and load CUDA kernels:

```python
# From setup.py (simplified)
from torch.utils.cpp_extension import CUDAExtension

ext_modules = [
    CUDAExtension(
        name="vllm._C",
        sources=[
            "csrc/cache_kernels.cu",
            "csrc/attention/paged_attention_v1.cu",
            "csrc/attention/paged_attention_v2.cu",
            "csrc/activation_kernels.cu",
            "csrc/layernorm_kernels.cu",
            "csrc/pos_encoding_kernels.cu",
            "csrc/pybind.cpp",  # Python bindings
        ],
        extra_compile_args={
            "cxx": ["-O3"],
            "nvcc": [
                "-O3",
                "--use_fast_math",
                "-gencode", "arch=compute_80,code=sm_80",  # A100
                "-gencode", "arch=compute_89,code=sm_89",  # L40
                "-gencode", "arch=compute_90,code=sm_90",  # H100
            ],
        },
    ),
]
```

### Python Bindings

```python
# From vllm/_custom_ops.py
def paged_attention_v1(
    out: torch.Tensor,
    query: torch.Tensor,
    key_cache: torch.Tensor,
    value_cache: torch.Tensor,
    num_kv_heads: int,
    scale: float,
    block_tables: torch.Tensor,
    context_lens: torch.Tensor,
    block_size: int,
    max_context_len: int,
    alibi_slopes: Optional[torch.Tensor],
) -> None:
    vllm_ops.paged_attention_v1(
        out, query, key_cache, value_cache,
        num_kv_heads, scale, block_tables,
        context_lens, block_size, max_context_len,
        alibi_slopes,
    )
```

In [ ]:
# Kernel compilation flags analysis

gpu_architectures = {
    'sm_70': 'V100 (Volta)',
    'sm_75': 'T4 (Turing)',
    'sm_80': 'A100 (Ampere)',
    'sm_86': 'A10G (Ampere)',
    'sm_89': 'L4/L40 (Ada Lovelace)',
    'sm_90': 'H100 (Hopper)',
}

features_by_arch = {
    'sm_70': ['FP16 Tensor Cores', 'HBM2'],
    'sm_75': ['FP16 Tensor Cores', 'INT8 Tensor Cores'],
    'sm_80': ['FP16/BF16 Tensor Cores', 'TF32', 'HBM2e'],
    'sm_86': ['FP16/BF16 Tensor Cores', 'Sparsity'],
    'sm_89': ['FP8 Tensor Cores', 'INT8', 'HBM3'],
    'sm_90': ['FP8 Tensor Cores', 'TMA', 'DPX', 'HBM3'],
}

print("GPU Architectures Supported by vLLM Kernels:")
print(f"{'Arch':<8} {'GPU':<25} {'Key Features'}")
print("-" * 70)
for arch, gpu in gpu_architectures.items():
    features = ', '.join(features_by_arch.get(arch, []))
    print(f"{arch:<8} {gpu:<25} {features}")

## 11. v1 vs v2 Selection Heuristic

vLLM chooses between PagedAttention v1 and v2 based on the context length:

```python
# From vllm/attention/backends/flash_attn.py (simplified)
# Use v2 when the number of KV blocks per sequence is large enough
# to benefit from partitioning

USE_V2_THRESHOLD = 8  # blocks per sequence

if max_context_len // block_size > USE_V2_THRESHOLD:
    paged_attention_v2(...)  # More parallelism for long sequences
else:
    paged_attention_v1(...)  # Less overhead for short sequences
```

In [ ]:
# Analyze when v2 is better than v1

def estimate_attention_time(
    version: str,
    context_len: int,
    num_heads: int = 32,
    head_dim: int = 128,
    block_size: int = 16,
    num_sms: int = 108,  # A100
    partition_size: int = 512,  # tokens per partition for v2
) -> float:
    """Estimate relative attention time for v1 vs v2."""
    num_blocks = (context_len + block_size - 1) // block_size
    
    if version == 'v1':
        # One thread block per (seq, head)
        # Thread block processes all KV blocks sequentially
        thread_blocks = num_heads
        work_per_tb = num_blocks
        
        # Time limited by the slowest thread block
        waves = max(1, thread_blocks / num_sms)
        time = work_per_tb / waves
    else:  # v2
        # Multiple thread blocks per (seq, head)
        num_partitions = (num_blocks + partition_size // block_size - 1) // (partition_size // block_size)
        thread_blocks = num_heads * num_partitions
        work_per_tb = partition_size // block_size
        
        # Better GPU utilization with more thread blocks
        waves = max(1, thread_blocks / num_sms)
        time = work_per_tb / waves + 0.1 * num_partitions  # Reduction overhead
    
    return time

context_lens = [64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]
v1_times = [estimate_attention_time('v1', cl) for cl in context_lens]
v2_times = [estimate_attention_time('v2', cl) for cl in context_lens]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(context_lens, v1_times, 'ro-', label='PagedAttention v1', linewidth=2)
ax.plot(context_lens, v2_times, 'bo-', label='PagedAttention v2', linewidth=2)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Length', fontsize=12)
ax.set_ylabel('Relative Time', fontsize=12)
ax.set_title('PagedAttention v1 vs v2 Performance', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Mark crossover point
for i, (v1, v2) in enumerate(zip(v1_times, v2_times)):
    if v2 < v1 and (i == 0 or v2_times[i-1] >= v1_times[i-1]):
        ax.axvline(x=context_lens[i], color='green', linestyle='--', 
                   label=f'Crossover: {context_lens[i]} tokens')
        ax.legend(fontsize=11)
        break

plt.tight_layout()
plt.savefig('/tmp/v1_vs_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

### Custom Kernel Overview

| Kernel | Purpose | Key Optimization |
|--------|---------|------------------|
| PagedAttention v1 | Single-pass attention over paged KV | Online softmax, block table lookup |
| PagedAttention v2 | Multi-partition attention | Better GPU utilization for long seqs |
| reshape_and_cache | Write KV to paged cache | Fused slot mapping + copy |
| copy_blocks | Block copy for beam search | Batched copy across layers |
| silu_and_mul | Fused SiLU activation | Saves 50% memory bandwidth |
| rms_norm | Fused RMSNorm | Single-pass reduction + normalize |
| rotary_embedding | Fused RoPE | Avoid materializing cos/sin tables |

## Exercises

### Exercise 1: Benchmark PagedAttention
Implement a benchmark that compares PagedAttention (v1 reference) against standard attention for varying context lengths. Measure the overhead of non-contiguous memory access.

### Exercise 2: Implement a Cache Copy Kernel
Write a reference implementation of the `copy_blocks` kernel that copies blocks across layers for beam search.

### Exercise 3: Fused QKV+RoPE
Design a fused kernel that combines QKV linear projection with rotary embedding. Analyze the memory bandwidth savings.

### Exercise 4: Block Size Impact on Kernel Performance
Analyze how block size affects PagedAttention kernel performance. Consider:
- Shared memory usage
- Register pressure
- Memory coalescing

In [ ]:
# Exercise 2 Starter: copy_blocks kernel

def copy_blocks_reference(
    key_caches: List[torch.Tensor],     # [num_layers] of [num_blocks, block_size, num_heads, head_dim]
    value_caches: List[torch.Tensor],   # same
    block_mapping: torch.Tensor,        # [num_pairs, 2] - (src, dst) pairs
):
    """
    Copy blocks for beam search fork.
    
    Mirrors vllm/csrc/cache_kernels.cu -> copy_blocks_kernel
    
    When a beam is forked, we need to copy the last block
    (which may diverge) to a new physical location.
    """
    num_layers = len(key_caches)
    num_pairs = block_mapping.shape[0]
    
    for layer_idx in range(num_layers):
        for pair_idx in range(num_pairs):
            src_block = block_mapping[pair_idx, 0].item()
            dst_block = block_mapping[pair_idx, 1].item()
            
            key_caches[layer_idx][dst_block].copy_(key_caches[layer_idx][src_block])
            value_caches[layer_idx][dst_block].copy_(value_caches[layer_idx][src_block])
    
    return key_caches, value_caches

# Test
num_layers = 4
num_blocks = 20
block_size = 4
num_heads = 4
head_dim = 32

k_caches = [torch.randn(num_blocks, block_size, num_heads, head_dim) for _ in range(num_layers)]
v_caches = [torch.randn(num_blocks, block_size, num_heads, head_dim) for _ in range(num_layers)]

# Copy blocks: (src=5, dst=10), (src=6, dst=11)
block_mapping = torch.tensor([[5, 10], [6, 11]])

print(f"Before copy:")
print(f"  K[0][5] sum: {k_caches[0][5].abs().sum():.4f}")
print(f"  K[0][10] sum: {k_caches[0][10].abs().sum():.4f}")

copy_blocks_reference(k_caches, v_caches, block_mapping)

print(f"\nAfter copy:")
print(f"  K[0][5] == K[0][10]: {torch.allclose(k_caches[0][5], k_caches[0][10])}")